# 실습 8: 기준 모델과 추천 모델
- 상황: 아무것도 안 해도 93점이 나온다는 걸 알았다
- 목표: 비교할 기준을 먼저 만들고, 그 위에서 진짜 모델을 재본다

## Step 0. 앞 실습까지 재현하기

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("../../day02/lab06_clean-dataset/results/secom_clean.csv")

sensor_cols = df.columns.drop("result")
df[sensor_cols] = df[sensor_cols].fillna(df[sensor_cols].median())

df["불량여부"] = (df["result"] == "불량").astype(int)

X = df[sensor_cols]
y = df["불량여부"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("학습용:", X_train.shape[0], "건, 불량", int(y_train.sum()), "건")
print("시험용:", X_test.shape[0], "건, 불량", int(y_test.sum()), "건")

학습용: 1253 건, 불량 83 건
시험용: 314 건, 불량 21 건


---

## Step 1. 오늘 쓸 말 정리하기

### 용어 풀이 - 모델을 비교할 때 쓰는 말

| 말 | 뜻 |
|---|---|
| 기준 모델 | 학습을 전혀 하지 않고 늘 같은 답만 내놓는 모델. 비교의 바닥선이 된다 |
| 학습 | 답이 붙은 기록을 넣어 규칙을 찾게 하는 일 |
| 예측 | 처음 보는 기록에 답을 붙이는 일 |
| 정확도 | 전체 중 맞힌 비율. 오늘 쓰는 유일한 점수이고, 내일 이 점수를 의심하게 된다 |

---

## Step 2. 게으름뱅이 모델 만들기

In [3]:
# numpy - 숫자 묶음을 다루는 도구를 np라는 짧은 이름으로 불러온다
import numpy as np

# 시험용 개수만큼 전부 0(양품)으로 채운 답안지를 만든다. 학습은 하지 않았다
기준예측 = np.zeros(len(y_test), dtype=int)

# 맞힌 개수 ÷ 전체 개수
기준정확도 = (기준예측 == y_test).mean()

print("기준 모델이 불량이라 한 건수:", 기준예측.sum())
print("기준 모델 정확도:", round(기준정확도 * 100, 2), "%")

기준 모델이 불량이라 한 건수: 0
기준 모델 정확도: 93.31 %


---

# Step 3. 왜 93점이 나오나

In [4]:
# 시험용에서 양품이 몇 건, 불량이 몇 건인지
print("시험용 양품:", (y_test == 0).sum(), "건")
print("시험용 불량:", (y_test == 1).sum(), "건")

# 전부 양품이라 답하면 -> 양품은 다 맞고, 불량은 다 틀린다
print("맞힌 것:", (y_test == 0).sum(), "/", len(y_test))

시험용 양품: 293 건
시험용 불량: 21 건
맞힌 것: 293 / 314


[기준 모델이 높은 점수를 받는 이유]<br>
시험용 [314]건 중 양품이 [293]건이다.<br>
전부 양품이라 답하면 [293]건은 자동으로 맞는다.<br>
불량 [21]건은 전부 놓치지만, 개수가 적어 점수에 거의 영향이 없다.

---

## Step 4. 모델을 추천받기

**1. 결정트리 (DecisionTreeClassifier)**
어떤 센서가 어떤 값을 넘었을 때 불량으로 갈렸는지 규칙(if-then) 형태로 바로 보여줄 수 있어서, 처음 결과를 설명하기에 가장 직관적입니다.
→ 단위를 맞출 필요 없음. 각 열을 독립적으로 임계값 기준으로 나누는 방식이라 열마다 자릿수가 달라도 분할에 영향이 없습니다.

**2. 로지스틱 회귀 (LogisticRegression)**
각 센서가 불량 확률을 올리는지 내리는지, 그 영향의 방향과 크기를 계수(coefficient) 하나로 바로 읽을 수 있어서 설명이 쉽습니다.
→ 단위를 맞춰야 함. 계수 크기가 그 열의 값 범위에 좌우되기 때문에(자릿수 큰 열이 부당하게 커 보임), StandardScaler 등으로 스케일을 맞춘 뒤 계수를 비교해야 의미가 있습니다. (지난 lab07에서 스케일 없이 돌렸을 때 수렴 경고가 뜨고 전부 양품으로만 예측된 것도 이 문제입니다.)

오늘은 두 모델 다 클래스 불균형 관련 옵션(class_weight 등)은 넣지 않고 기본값 그대로 쓰면 됩니다.

**[추천받은 모델]**<br>
1. [로지스틱 회귀] - [둘 중 하나를 고르는 문제의 기본이고, 어느 열이 얼마나 작용했는지 볼 수 있다]<br>
2. [의사결정나무] - [자르는 기준이 눈에 보여서 설명하기 쉽다]<br>
내가 고른 것 : [로지스틱 회귀]

---

# Step 5: 추천 모델 학습시키고 점수 재기

In [5]:
# 로지스틱 회귀는 단위를 맞춰야 하므로 StandardScaler로 표준화한다
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# class_weight 등 불균형 보정 없이 기본 설정 그대로 사용한다
model = LogisticRegression()
model.fit(X_train_scaled, y_train)

예측 = model.predict(X_test_scaled)

정확도 = (예측 == y_test).mean() * 100
불량예측건수 = int((예측 == 1).sum())
불량예측중_실제불량 = int(((예측 == 1) & (y_test == 1)).sum())

print("정확도:", round(정확도, 2), "%")
print("불량이라고 예측한 건수:", 불량예측건수)
print("그중 실제로 불량이었던 건수:", 불량예측중_실제불량)

정확도: 92.99 %
불량이라고 예측한 건수: 5
그중 실제로 불량이었던 건수: 2


---

## Step 6. 모델 기록표

| 모델 | 왜 썼나 | 정확도 | 불량이라 예측한 건수 | 그중 진짜 |
|---|---|---|---|---|
| 기준 모델 (전부 양품) | 비교할 바닥선 | [93.31]% | [0] | [0] |
| [로지스틱 회귀] | [분류의 기본이고 결과를 설명하기 쉬워서] | [92.99]% | [5] | [2] |

---
## 직접 해보기 (도전) - 게으름뱅이를 반대로 만들면

- 상황: 전부 양품이라 답하는 모델을 만들어봤다. 반대는 어떨까
- 할 일: 전부 불량이라 답하는 모델의 점수를 재고, 추천 모델을 하나 더 붙여 표를 늘린다
- 결과물: 네 줄짜리 기록표 1개

In [6]:
# numpy - 숫자 묶음을 다루는 도구를 np라는 짧은 이름으로 불러온다
import numpy as np

# 시험용 개수만큼 전부 0(양품)으로 채운 답안지를 만든다. 학습은 하지 않았다
기준예측 = np.ones(len(y_test), dtype=int)

# 맞힌 개수 ÷ 전체 개수
기준정확도 = (기준예측 == y_test).mean()

print("기준 모델이 불량이라 한 건수:", 기준예측.sum())
print("기준 모델 정확도:", round(기준정확도 * 100, 2), "%")

기준 모델이 불량이라 한 건수: 314
기준 모델 정확도: 6.69 %


In [7]:
# Step 4에서 추천받은 두 모델 중 아직 안 써본 결정트리를 학습시킨다
from sklearn.tree import DecisionTreeClassifier

# 결정트리는 열마다 자릿수가 달라도 상관없어 스케일링이 필요 없다
# class_weight 등 불균형 보정 없이 기본 설정 그대로 사용한다
tree_model = DecisionTreeClassifier(random_state=42)
tree_model.fit(X_train, y_train)

트리예측 = tree_model.predict(X_test)

def 평가(pred):
    정확도 = round((pred == y_test).mean() * 100, 2)
    불량건수 = int((pred == 1).sum())
    진짜불량 = int(((pred == 1) & (y_test == 1)).sum())
    return 정확도, 불량건수, 진짜불량

# 기존 X_train, X_test, y_train, y_test, model, 예측 등은 그대로 두고
# 비교표를 만들기 위한 값만 새 이름으로 계산한다
예측_전부양품 = np.zeros(len(y_test), dtype=int)
예측_전부불량 = np.ones(len(y_test), dtype=int)

기록 = {
    "기준 모델 (전부 양품)": 평가(예측_전부양품),
    "전부 불량 모델": 평가(예측_전부불량),
    "로지스틱 회귀 (앞에서 학습)": 평가(예측),
    "결정트리 (방금 학습)": 평가(트리예측),
}

기록표 = pd.DataFrame(
    [
        {"모델": name, "정확도(%)": acc, "불량이라 한 건수": cnt, "그중 진짜": hit}
        for name, (acc, cnt, hit) in 기록.items()
    ]
).set_index("모델")

기록표

,정확도(%),불량이라 한 건수,그중 진짜
모델,,,
기준 모델 (전부 양품),93.31,0,0
전부 불량 모델,6.69,314,21
로지스틱 회귀 (앞에서 학습),92.99,5,2
결정트리 (방금 학습),84.71,33,3


### 네 모델 비교

| 모델 | 정확도 | 불량이라 한 건수 | 그중 진짜 |
|---|---|---|---|
| 전부 양품 | [93.31]% | [0] | [0] |
| 전부 불량 | [6.69]% | [314] | [21] |
| [로지스틱 회귀] | [92.99]% | [5] | [2] |
| [의사결정나무] | [84.71]% | [33] | [3] |